
# GeoLife CP1 — Final Stay-point Sensitivity Validation

Companion notebook sau full-release baseline. Notebook này **reuse** `baseline_summary_v2.pkl`, không chạy lại 18,670 trajectories.

Mục tiêu:
- smoke-check complete cleaning audit events;
- inspect hard-speed outliers từ baseline cache;
- chạy 27 configs trên deterministic user-stratified sample;
- thêm user-level recurring-location stability proxy trước khi freeze CP1 baseline.


In [ ]:

from pathlib import Path
from time import perf_counter
from IPython.display import display
import os, subprocess, sys
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp1-cleaning-staypoint")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
CACHE_DIR = Path(os.environ.get("GEOLIFE_CACHE_DIR", "/mnt/geolife-data/cache/cp1_staypoints"))

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        Path("/mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data"),
        Path("/mnt/geolife-data/Data"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(Path("/mnt/geolife-data").glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
sys.path.insert(0, str(REPO_DIR / "src"))

from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, clean_trajectory_with_audit, detect_staypoints

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

def read_plt(path):
    df = pd.read_csv(
        path, skiprows=6, header=None,
        names=["latitude","longitude","unused","altitude","date_days","date","time"],
    )
    df["timestamp"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
        utc=True,
    )
    if df["timestamp"].isna().any():
        raise ValueError(f"Unparsable timestamp in {path}")
    return df[["timestamp","latitude","longitude"]]

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("DATA_ROOT:", DATA_ROOT)
print("Trajectory files:", len(files))



## 1. Reuse full-release baseline + audit reconciliation


In [ ]:

BASELINE_CACHE = CACHE_DIR / "baseline_summary_v2.pkl"
if not BASELINE_CACHE.exists():
    raise FileNotFoundError(
        f"{BASELINE_CACHE} not found. Run notebook 02 full baseline first."
    )

baseline_summary = pd.read_pickle(BASELINE_CACHE)
if len(baseline_summary) != len(files):
    raise RuntimeError(
        f"Baseline cache rows={len(baseline_summary):,}, expected={len(files):,}"
    )

print("Total stays:", int(baseline_summary["n_stays"].sum()))
print("Files with >=1 stay:", int((baseline_summary["n_stays"] > 0).sum()))
print("Attached conflict reasons:", int(baseline_summary["same_second_conflict_boundaries"].sum()))
print("EDA exact reference: same-second >10m ambiguity groups=835, invalid points=1")

display(
    baseline_summary.nlargest(10, "hard_speed_boundaries")[
        ["file","raw_rows","clean_rows","n_sequences","n_stays",
         "temporal_gap_boundaries","hard_speed_boundaries"]
    ]
)

# Terminal conflict: no later retained row exists, but event must remain observable.
probe = pd.DataFrame(
    [
        ("2026-01-01T09:59:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.02, 116.0),
    ],
    columns=["timestamp","latitude","longitude"],
)
probe["timestamp"] = pd.to_datetime(probe["timestamp"], utc=True)
cleaned_probe, audit_probe = clean_trajectory_with_audit(probe)
display(audit_probe)
assert (audit_probe["reason"] == "same_second_spatial_ambiguity").sum() == 1
print("Complete audit-event smoke check: OK")



## 2. Deterministic user-stratified sample

Mỗi user đóng góp tối đa 5 trajectories. Heavy users được sample trải đều trên sorted history thay vì lấy prefix.


In [ ]:

MAX_FILES_PER_USER = 5

def user_id_from_path(path):
    return path.parent.parent.name

def build_user_stratified_sample(paths, max_files_per_user=5):
    by_user = {}
    for path in paths:
        by_user.setdefault(user_id_from_path(path), []).append(path)

    sample = []
    for user_id in sorted(by_user):
        user_paths = sorted(by_user[user_id])
        k = min(max_files_per_user, len(user_paths))
        if k == len(user_paths):
            selected = user_paths
        else:
            idx = np.unique(np.linspace(0, len(user_paths) - 1, num=k, dtype=int))
            selected = [user_paths[int(i)] for i in idx]
        sample.extend((user_id, path) for path in selected)
    return sample

sample = build_user_stratified_sample(files, MAX_FILES_PER_USER)
manifest = pd.DataFrame([{"user_id": u, "file": str(p)} for u,p in sample])

print("Users covered:", manifest["user_id"].nunique())
print("Files sampled:", len(manifest))
display(manifest.groupby("user_id").size().rename("sampled_files").describe())
assert manifest["user_id"].nunique() == len({user_id_from_path(p) for p in files})



## 3. 27-config sensitivity with user-level recurrence proxy

`repeat_location_user_rate`: among users with >=2 detected stays, fraction having at least one pair of stay representatives within 200 m.


In [ ]:

GAPS = [120, 300, 600]
DISTANCES = [100, 200, 300]
DWELLS = [600, 1200, 1800]
REPEAT_RADIUS_M = 200.0
SENSITIVITY_CACHE = CACHE_DIR / "sensitivity_user_stratified_v1.pkl"

grid = [(g,d,w) for g in GAPS for d in DISTANCES for w in DWELLS]

def repeat_metrics(stays_by_user):
    users_2plus = repeat_users = 0
    for coords_list in stays_by_user.values():
        if len(coords_list) < 2:
            continue
        users_2plus += 1
        coords = np.asarray(coords_list, dtype=float)
        hit = False
        for i in range(len(coords) - 1):
            distances = np.asarray(
                haversine_m(
                    coords[i,0], coords[i,1],
                    coords[i+1:,0], coords[i+1:,1],
                ),
                dtype=float,
            )
            if np.any(distances <= REPEAT_RADIUS_M):
                hit = True
                break
        repeat_users += int(hit)
    return users_2plus, repeat_users, (
        repeat_users / users_2plus if users_2plus else np.nan
    )

def evaluate_grid(sample):
    accum = {
        key: {"n_stays":0, "files_with_stays":0, "durations":[], "by_user":{}}
        for key in grid
    }
    configs_by_gap = {
        gap: [(d,w) for g,d,w in grid if g == gap]
        for gap in GAPS
    }

    t0 = perf_counter()
    for i, (user_id, path) in enumerate(sample, 1):
        raw = read_plt(path)
        for gap in GAPS:
            cleaned = clean_trajectory(
                raw,
                same_second_radius_m=BASELINE["same_second_radius_m"],
                max_gap_s=gap,
                hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
            )
            for distance_m, dwell_s in configs_by_gap[gap]:
                bucket = accum[(gap,distance_m,dwell_s)]
                stays = detect_staypoints(
                    cleaned,
                    distance_threshold_m=distance_m,
                    min_dwell_s=dwell_s,
                )
                if stays.empty:
                    continue
                bucket["n_stays"] += len(stays)
                bucket["files_with_stays"] += 1
                bucket["durations"].extend(stays["duration_s"].astype(float).tolist())
                bucket["by_user"].setdefault(user_id, []).extend(
                    stays[["latitude","longitude"]].to_numpy(dtype=float).tolist()
                )
        if i % 100 == 0:
            print(f"{i:,}/{len(sample):,} files | {(perf_counter()-t0)/60:.1f} min")

    users = sorted({u for u,_ in sample})
    rows = []
    for gap,distance_m,dwell_s in grid:
        bucket = accum[(gap,distance_m,dwell_s)]
        counts = np.array([len(bucket["by_user"].get(u, [])) for u in users], dtype=float)
        active = counts[counts > 0]
        durations = bucket["durations"]
        users_2plus, repeat_users, repeat_rate = repeat_metrics(bucket["by_user"])
        rows.append({
            "max_gap_s": gap,
            "distance_threshold_m": distance_m,
            "min_dwell_s": dwell_s,
            "n_stays": bucket["n_stays"],
            "files_with_stays": bucket["files_with_stays"],
            "users_with_stays": int((counts > 0).sum()),
            "mean_stays_per_user": float(counts.mean()),
            "median_stays_per_active_user": float(np.median(active)) if active.size else np.nan,
            "users_with_2plus_stays": users_2plus,
            "repeat_location_users": repeat_users,
            "repeat_location_user_rate": repeat_rate,
            "median_duration_s": float(np.median(durations)) if durations else np.nan,
            "p90_duration_s": float(np.quantile(durations, 0.9)) if durations else np.nan,
        })
    return pd.DataFrame(rows)

if SENSITIVITY_CACHE.exists():
    sensitivity = pd.read_pickle(SENSITIVITY_CACHE)
    print("Loaded:", SENSITIVITY_CACHE)
else:
    sensitivity = evaluate_grid(sample)
    sensitivity.to_pickle(SENSITIVITY_CACHE)
    print("Saved:", SENSITIVITY_CACHE)

display(sensitivity.sort_values(["max_gap_s","distance_threshold_m","min_dwell_s"]))

baseline_row = sensitivity[
    (sensitivity["max_gap_s"] == 300)
    & (sensitivity["distance_threshold_m"] == 200)
    & (sensitivity["min_dwell_s"] == 1200)
]
print("Baseline config:")
display(baseline_row)



## 4. Review gate

Không freeze chỉ vì baseline tạo ra số stay “vừa phải”. Review:

- baseline có nằm giữa vùng sensitivity thay vì extreme không;
- user coverage có thay đổi mượt khi đổi threshold không;
- recurring-location proxy có ổn định quanh baseline không;
- hard-speed outlier có giải thích được như corruption/interleaving không.

Sau khi review bảng này: ghi quyết định baseline vào `docs/`, merge PR #3, rồi bắt đầu Home/Office notebook.
